In [15]:
import pandas as pd
import numpy as np
from sklearn.linear_model import LinearRegression, Ridge
from sklearn.model_selection import train_test_split
from sklearn.metrics import r2_score, mean_squared_error
from sklearn.preprocessing import PolynomialFeatures, StandardScaler
from sklearn.pipeline import Pipeline
import sys
from sklearn.ensemble import RandomForestRegressor
import lightgbm as lgb
from sklearn.model_selection import GridSearchCV

In [16]:
df = pd.read_csv('orange_df.csv')
df.head()

,품종,지역,측정 날짜,평균 당도,씨앗,산도,당산비,과일 갯수,평균 너비,장폭비,...,길이,너비,줄리안 날짜,대목,수령,당도1,당도2,산도1,산도2,평균 산도 측정
0,Aoshima,Lindcove,2005-10-06,8.2,50.0,1.55,5.33,10,6.19,0.84,...,52.0,61.9,NaN,16-6,8,8.2,8.2,9.19,9.14,9.165
1,Aoshima,Lindcove,2005-10-06,7.9,5.0,1.34,5.90,10,6.84,0.87,...,59.2,68.4,NaN,CZO,9,7.9,7.9,7.92,7.92,7.920
2,Aoshima,Lindcove,2005-10-20,9.1,27.0,1.28,7.17,10,6.55,0.83,...,54.6,65.5,NaN,16-6,8,9.1,9.1,4.19,4.16,4.175
3,Aoshima,Lindcove,2005-10-20,9.1,34.0,1.20,7.61,10,6.75,0.83,...,56.0,67.5,NaN,CZO,9,9.1,9.1,3.92,3.94,3.930
4,Aoshima,Lindcove,2005-11-18,10.1,37.0,0.96,10.52,10,7.17,0.82,...,58.7,71.7,NaN,16-6,8,10.1,10.1,3.79,3.72,3.755


# 데이터 전처리

In [17]:
try:
    df = pd.read_csv('orange_df.csv')
except FileNotFoundError:
    print("Error: orange_df.csv not found. Please ensure the file is in the same directory as the script.")
    sys.exit() # exit() 대신 sys.exit() 권장

target_variable = '평균 당도'

if target_variable not in df.columns:
    print(f"Error: Target variable '{target_variable}' not found in the DataFrame columns. Available columns: {df.columns.tolist()}")
    sys.exit()

df.dropna(subset=[target_variable], inplace=True)
numeric_df = df.select_dtypes(include=np.number)

if target_variable not in numeric_df.columns:
    print(f"Error: Target variable '{target_variable}' is not numeric and cannot be used for correlation.")
    sys.exit()

# 상관관계 계산
correlations_series = numeric_df.corr()[target_variable]

# 피처에서 제외할 변수 리스트
cols_to_exclude = ['당도1', '당도2', '산도1', '산도2', '평균 산도', '율리우스일', '당도 산도 비율', 'Citric acid (%)', target_variable]
existing_cols_to_exclude = [col for col in cols_to_exclude if col in correlations_series.index]
correlations = correlations_series.drop(existing_cols_to_exclude)

# 0.2 '이상'인 변수들만 사용
selected_features = correlations[np.abs(correlations) >= 0.2].index.tolist()
drop_columns = "평균 무게"  # 제외하고자 하는 컬럼 작성, 없을 시 주석 처리
selected_features = [item for item in selected_features if item not in drop_columns]

if not selected_features:
    print("No numeric features found with absolute correlation to '당도' >= 0.2. Cannot build a model.")
    sys.exit()

print(f"Selected features based on correlation with '{target_variable}' (abs correlation >= 0.2):")
for feature in selected_features:
    print(f"- {feature}: {correlations[feature]:.4f}")

X = df[selected_features]
y = df[target_variable]

# X의 NaN을 처리한 후, 해당 인덱스를 사용해 y를 필터링
X = X.dropna()
y = y[X.index]

if X.empty:
    print("After dropping NaN values, no valid data remains for selected features. Cannot build a model.")
    sys.exit()

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

Selected features based on correlation with '평균 당도' (abs correlation >= 0.2):
- 산도: 0.2313
- 당산비: 0.3076
- 평균 너비: -0.2602
- 색상: 0.3099
- 과즙 비율: 0.2044
- 평균길이: -0.2758
- 줄리안 날짜: 0.3181
- 평균 산도 측정: 0.3320


#linear regression

In [18]:
model = LinearRegression()
model.fit(X_train, y_train)

y_pred = model.predict(X_test)

r2 = r2_score(y_test, y_pred)
mse = mean_squared_error(y_test, y_pred)

print(f"\nModel R-squared score: {r2:.4f}")
print(f"Model Mean Squared Error (MSE): {mse:.4f}")

print("\nModel Coefficients:")
for feature, coef in zip(selected_features, model.coef_):
    print(f"- {feature}: {coef:.4f}")
print(f"Intercept: {model.intercept_:.4f}")


Model R-squared score: 0.8509
Model Mean Squared Error (MSE): 0.4921

Model Coefficients:
- 산도: -118.4269
- 당산비: 0.2589
- 평균 너비: -0.0055
- 색상: 0.0357
- 과즙 비율: 0.0304
- 평균길이: -0.0534
- 줄리안 날짜: 0.0017
- 평균 산도 측정: 29.2625
Intercept: 1.7130


# polynomial regression

In [19]:
degree = 2

params = {
    'poly__degree': [2, 3]
}

model = Pipeline([
    ("poly", PolynomialFeatures(degree=degree, include_bias=False)),
    ("scaler", StandardScaler()), # 다항 특성은 값의 범위가 매우 커질 수 있으므로 스케일링 권장
    ("lin_reg", LinearRegression())
])

grid_search = GridSearchCV(
    estimator=model,
    param_grid=params,
    cv=3,
    scoring=['neg_mean_squared_error', 'r2'],
    n_jobs=-1,
    refit='r2'
)

# 모델 훈련 (파이프라인이 알아서 변환 및 훈련 진행)
grid_search.fit(X_train, y_train)

print(f"최적의 하이퍼파라미터 (best_params): {grid_search.best_params_}")
print(f"최고 교차 검증 점수 (best_score): {grid_search.best_score_}")
best_model = grid_search.best_estimator_

# 평가
r2 = r2_score(y_test, y_pred)
mse = mean_squared_error(y_test, y_pred)

print(f"\n--- Polynomial Regression (Degree={degree}) ---")
print(f"Model R-squared score: {r2:.4f}")
print(f"Model Mean Squared Error (MSE): {mse:.4f}")

# --- 계수 출력 부분 수정 ---
print("\nModel Coefficients (for polynomial features):")

try:
    # 파이프라인 내부의 'lin_reg' (LinearRegression) 단계의 계수
    coefficients = model.named_steps['lin_reg'].coef_

    # 파이프라인 내부의 'poly' (PolynomialFeatures) 단계에서 생성된 특성 이름
    # X_train.columns (pandas) 대신 selected_features (list)를 전달
    feature_names = model.named_steps['poly'].get_feature_names_out(selected_features)

    # y가 1차원이므로 model.coef_는 (n_features,) 형태일 수 있음
    # 만약 (1, n_features) 형태라면 [0]을 사용
    if coefficients.ndim > 1:
        coefficients = coefficients[0]

    for feature, coef in zip(feature_names, coefficients):
        print(f"- {feature}: {coef:.4f}")

    # 절편(Intercept) 접근
    intercept = model.named_steps['lin_reg'].intercept_
    # intercept가 스칼라가 아닐 경우 (보통 1차원 배열) [0]으로 접근
    if hasattr(intercept, "__len__"):
        intercept = intercept[0]

    print(f"Intercept: {intercept:.4f}")

except Exception as e:
    print(f"Error printing coefficients: {e}")
    print("Note: Coefficient printing relies on .get_feature_names_out() (scikit-learn 1.0+).")
    print("Coefficients are inside model.named_steps['lin_reg'].coef_")

최적의 하이퍼파라미터 (best_params): {'poly__degree': 2}
최고 교차 검증 점수 (best_score): 0.9994771258697949

--- Polynomial Regression (Degree=2) ---
Model R-squared score: 0.8509
Model Mean Squared Error (MSE): 0.4921

Model Coefficients (for polynomial features):
Error printing coefficients: 'LinearRegression' object has no attribute 'coef_'
Note: Coefficient printing relies on .get_feature_names_out() (scikit-learn 1.0+).
Coefficients are inside model.named_steps['lin_reg'].coef_


#Decision Tree

In [20]:
params = {
    'n_estimators': [100, 200, 500],
    'max_depth': [3, 5, 10],
}

model = RandomForestRegressor(random_state=42, n_jobs=-1)

grid_search = GridSearchCV(
    estimator=model,
    param_grid=params,
    cv=3,
    scoring='neg_mean_squared_error',
    n_jobs=-1,
    verbose=1 # (로그 출력)
)

# 모델 훈련 (파이프라인이 알아서 변환 및 훈련 진행)
grid_search.fit(X_train, y_train)

# 예측
print(f"최적의 하이퍼파라미터 (best_params): {grid_search.best_params_}")
print(f"최고 교차 검증 점수 (best_score): {grid_search.best_score_}")
best_model = grid_search.best_estimator_

# 평가
r2 = r2_score(y_test, y_pred)
mse = mean_squared_error(y_test, y_pred)

# --- (수정) 모델 이름 변경 ---
print(f"\n--- Random Forest Regressor ---")
print(f"Model R-squared score: {r2:.4f}")
print(f"Model Mean Squared Error (MSE): {mse:.4f}")

# --- (수정) 계수 대신 '특성 중요도' 출력 ---
print("\nModel Feature Importances:")

try:
    # RandomForest는 .coef_ 대신 .feature_importances_ 를 사용
    importances = model.feature_importances_

    # selected_features는 원본 특성 이름 (RandomForest는 변환이 필요 없음)
    for feature, importance in zip(selected_features, importances):
        print(f"- {feature}: {importance:.4f}")

    # RandomForest는 선형 모델이 아니므로 'Intercept' (절편)이 없습니다.
    # intercept = model.named_steps['lin_reg'].intercept_
    # ...
    # print(f"Intercept: {intercept:.4f}")

except Exception as e:
    print(f"Error printing feature importances: {e}")
    print("Coefficients are available in .feature_importances_ for Random Forest.")

Fitting 3 folds for each of 9 candidates, totalling 27 fits
최적의 하이퍼파라미터 (best_params): {'max_depth': 10, 'n_estimators': 500}
최고 교차 검증 점수 (best_score): -0.40835240572907044

--- Random Forest Regressor ---
Model R-squared score: 0.8509
Model Mean Squared Error (MSE): 0.4921

Model Feature Importances:
Error printing feature importances: This RandomForestRegressor instance is not fitted yet. Call 'fit' with appropriate arguments before using this estimator.
Coefficients are available in .feature_importances_ for Random Forest.


# RidgeRegression

In [21]:
model = Ridge(alpha=1.0)
model.fit(X_train, y_train)

y_pred = model.predict(X_test)

r2 = r2_score(y_test, y_pred)
mse = mean_squared_error(y_test, y_pred)

print(f"\nModel R-squared score: {r2:.4f}")
print(f"Model Mean Squared Error (MSE): {mse:.4f}")

print("\nModel Coefficients:")
for feature, coef in zip(selected_features, model.coef_):
    print(f"- {feature}: {coef:.4f}")
print(f"Intercept: {model.intercept_:.4f}")


Model R-squared score: 0.7147
Model Mean Squared Error (MSE): 0.9417

Model Coefficients:
- 산도: -2.2927
- 당산비: 0.4726
- 평균 너비: -0.2076
- 색상: 0.0494
- 과즙 비율: 0.0410
- 평균길이: -0.0514
- 줄리안 날짜: 0.0082
- 평균 산도 측정: 1.6765
Intercept: -3.5080


#gradiant boosting

In [22]:
model = lgb.LGBMRegressor(n_estimators=100, learning_rate=0.01, random_state=42, verbose=-1)

model.fit(X_train, y_train)
y_pred = model.predict(X_test)

r2 = r2_score(y_test, y_pred)
mse = mean_squared_error(y_test, y_pred)

print(f"\nModel R-squared score: {r2:.4f}")
print(f"Model Mean Squared Error (MSE): {mse:.4f}")


Model R-squared score: 0.6214
Model Mean Squared Error (MSE): 1.2495


In [ ]:
# 1. 사용자 정보 설정
!git config --global user.email "taehan3303@gmail.com"
!git config --global user.name "taehan"

# 2. 저장소 클론 (이미 되어 있다면 패스)
# 토큰을 주소에 포함해야 권한 에러가 나지 않습니다.
!git clone https://{내_토큰}@github.com/{taehan}/{저장소명}.git

# 3. 변경 사항 반영 및 Push
%cd {저장소명}
!git add .
!git commit -m "Colab에서 수정함"
!git push origin main